In [27]:
import pandas as pd
import numpy as np
import json

from dbm301.config import RAW_DATA_DIR, FIGURES_DIR, PROCESSED_DATA_DIR ,INTERIM_DATA_DIR 

# Tạo thư mục output nếu chưa có
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
INTERIM_DATA_DIR.mkdir(parents=True, exist_ok=True)

# ✅ CÓ sep=';' để chia đúng 17 cột
df_exploration = pd.read_csv(INTERIM_DATA_DIR / 'exploration_80.csv', sep=';')
df_test = pd.read_csv(INTERIM_DATA_DIR / 'test_20.csv', sep=';')


2.6

In [28]:
# TASK 2.6: ĐỊNH NGHĨA VÀ LƯU CẤU HÌNH RANH GIỚI (BINNING CONFIG) TỪ TẬP 80%

# 1. Xác định các mốc cắt (ranh giới) dựa trên phân tích ở Task 2.3
binning_config = {
    'age': {
        'bins': [0, 29, 59, 120],
        'labels': ['young', 'middle_age', 'senior']
    },
    'balance': {
        'bins': [-float('inf'), -0.001, 500, 2000, float('inf')],
        'labels': ['negative', 'low', 'medium', 'high']
    },
    'campaign': {
        'bins': [0, 1, 3, float('inf')],
        'labels': ['1_call', '2_3_calls', '4plus_calls']
    }
}

# 2. Lưu cấu hình ra file JSON để làm bằng chứng nộp bài (Task 2.6)
with open(PROCESSED_DATA_DIR / 'binning_config.json', 'w', encoding='utf-8') as f:
    json.dump(binning_config, f, indent=4, ensure_ascii=False)

print("✅ TASK 2.6 HOÀN THÀNH: Đã lưu cấu hình ranh giới vào " + str(PROCESSED_DATA_DIR / 'binning_config.json') + "!")

✅ TASK 2.6 HOÀN THÀNH: Đã lưu cấu hình ranh giới vào C:\Users\Admin\Documents\DBM301\Assignment1_de4\dbm301_asm_group3_de4\data\processed\binning_config.json!


2.7

In [29]:
# TASK 2.7: HÀM BIẾN ĐỔI DỮ LIỆU CHUẨN HÓA
def apply_discretization(data, config):
    df_clean = data.copy()
    
    # 1. Rời rạc hóa age, balance, campaign theo cấu hình đã học ở Task 2.6
    df_clean['age_group'] = pd.cut(
        df_clean['age'], 
        bins=config['age']['bins'], 
        labels=config['age']['labels']
    )
    df_clean['balance_group'] = pd.cut(
        df_clean['balance'], 
        bins=config['balance']['bins'], 
        labels=config['balance']['labels']
    )
    df_clean['campaign_group'] = pd.cut(
        df_clean['campaign'], 
        bins=config['campaign']['bins'], 
        labels=config['campaign']['labels']
    )
    
    # 2. Biến đổi pdays thành biến nhị phân (contacted_before)
    df_clean['contacted_before'] = df_clean['pdays'].apply(
        lambda x: 'no' if x == -1 else 'yes'
    )
    
    # 3. LOẠI BỎ: duration (tránh data leakage), day (khử nhiễu), và các cột số gốc cũ
    cols_to_drop = ['duration', 'day', 'age', 'balance', 'campaign', 'pdays']
    df_clean = df_clean.drop(columns=cols_to_drop)
    
    return df_clean

# Áp dụng cho tập 80% (Exploration)
processed_exploration = apply_discretization(df_exploration, binning_config)

# ÁP DỤNG NGUYÊN TRẠNG cho tập 20% (Test)
processed_test = apply_discretization(df_test, binning_config)

print(f"Số cột sau xử lý: {processed_exploration.shape[1]} cột (đã loại bỏ duration và day)")

Số cột sau xử lý: 15 cột (đã loại bỏ duration và day)


In [30]:
# Xem thử 5 dòng đầu tiên của tập dữ liệu đã làm sạch
print("=== 5 DÒNG ĐẦU TIÊN CỦA TẬP KHÁM PHÁ (80%) ===")
display(processed_exploration.head())

# Lưu 2 file dữ liệu sạch vào thư mục data/processed/
processed_exploration.to_csv(PROCESSED_DATA_DIR / 'processed_exploration.csv', sep=';', index=False)
processed_test.to_csv(PROCESSED_DATA_DIR / 'processed_test.csv', sep=';', index=False)

print("\n🎉 ĐÃ XUẤT XONG 2 FILE MINING CHUẨN VÀO THƯ MỤC data/processed/:")
print("1. data/processed/processed_exploration.csv (36,168 dòng)")
print("2. data/processed/processed_test.csv (9,043 dòng)")

=== 5 DÒNG ĐẦU TIÊN CỦA TẬP KHÁM PHÁ (80%) ===


,job,marital,education,default,housing,loan,contact,month,previous,poutcome,y,age_group,balance_group,campaign_group,contacted_before
0,technician,divorced,secondary,no,no,no,telephone,aug,0,unknown,no,middle_age,medium,2_3_calls,no
1,student,single,secondary,no,no,no,cellular,apr,7,failure,yes,young,high,4plus_calls,yes
2,technician,single,secondary,no,yes,no,cellular,aug,0,unknown,yes,middle_age,low,4plus_calls,no
3,unemployed,married,secondary,no,no,no,telephone,jul,0,unknown,no,middle_age,low,4plus_calls,no
4,technician,married,secondary,no,no,no,cellular,aug,0,unknown,no,middle_age,low,4plus_calls,no



🎉 ĐÃ XUẤT XONG 2 FILE MINING CHUẨN VÀO THƯ MỤC data/processed/:
1. data/processed/processed_exploration.csv (36,168 dòng)
2. data/processed/processed_test.csv (9,043 dòng)
